# **2일차 팀 프로젝트: 문서 기반 RAG 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 PDF 문서를 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 적용
3. 검색 테스트 및 RAG 시스템 구현

## 구현 단계
- 환경 설정 확인
- PDF 문서 로딩
- Child Chunk 생성 및 Qdrant Cloud 저장
- Parent Document 저장
- 검색 테스트
- RAG 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://2b5fb460-5417-4840-8cbb-f9716463ff29.eu-central-1-0.aws.cloud.qdrant.io


## 1. PDF 문서 로딩

**TODO: 팀에서 선정한 PDF 파일 경로를 입력하세요**

In [15]:
from langchain_core.documents import Document
import fitz

# TODO: PDF 파일 경로를 입력하세요
# 예시: "../datasets/your_document.pdf"
file_path = "C:/Users/USER/Desktop/smu0818/3._2026-2학기_개설학과별_시간표_2026.8.19._기준.pdf"

doc = fitz.open(file_path)
docs = []

# 페이지 단위로 Document 생성 (Parent Document)
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text", sort=True)

    # 빈 페이지는 스킵
    if len(text.strip()) < 10:
        continue

    docs.append(
        Document(
            page_content=text,
            metadata={
                "source": file_path.split("/")[-1],
                "page": page_num + 1,
                "parent_id": f"page_{page_num + 1}"
            }
        )
    )

doc.close()

print(f"총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

# 첫 페이지 내용 미리보기
print(f"\n첫 페이지 내용 미리보기:")
print(docs[0].page_content[:300] + "...")

총 103개의 페이지(Parent Document) 로드 완료

첫 번째 페이지 길이: 416자
평균 페이지 길이: 2341자

첫 페이지 내용 미리보기:
               2026학년도2학기학과별시간표(공지용)


글로벌인문학부대학
                                  실습             강의시간    분 No 학년이수  학수번호      교과목명     학점이론     교양영역                  담당교수        비고
      구분                       시간 시간                  (강의실)    반
               취업과창업(글로벌인문학부대학  1   3 1전선HBCD0001    ...


## 2. Child Chunk 생성

**TODO: 청킹 전략을 조정해보세요 (선택사항)**
- chunk_size: 각 청크의 크기 (기본 400자)
- chunk_overlap: 청크 간 겹치는 부분 (기본 50자)

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: 필요시 chunk_size와 chunk_overlap 값을 조정하세요
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # 작은 크기로 정확한 검색
    chunk_overlap=50     # 문맥 유지
)

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"],
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"]
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

# Child chunk 샘플 확인
print(f"\nChild chunk 샘플 (첫 3개):")
for i in range(min(3, len(child_docs))):
    print(f"\nChunk {i + 1}:")
    print(f"  Parent ID: {child_docs[i].metadata['parent_id']}")
    print(f"  Page: {child_docs[i].metadata['page']}")
    print(f"  Length: {len(child_docs[i].page_content)}자")
    print(f"  Content: {child_docs[i].page_content[:100]}...")


생성된 통계:
  - Parent 문서 수: 103
  - Child chunk 수: 914
  - 평균 chunk/page: 8.9

Child chunk 샘플 (첫 3개):

Chunk 1:
  Parent ID: page_1
  Page: 1
  Length: 21자
  Content: 2026학년도2학기학과별시간표(공지용)...

Chunk 2:
  Parent ID: page_1
  Page: 1
  Length: 377자
  Content: 글로벌인문학부대학
                                  실습             강의시간    분 No 학년이수  학수번호      교과목명     학점이...

Chunk 3:
  Parent ID: page_2
  Page: 2
  Length: 21자
  Content: 2026학년도2학기학과별시간표(공지용)...


## 3. Qdrant Cloud에 Child Chunk 저장

**TODO: 컬렉션 이름을 팀 프로젝트에 맞게 변경하세요**

In [17]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

Qdrant Cloud에 연결되었습니다.
  URL: https://2b5fb460-5417-4840-8cbb-f9716463ff29.eu-central-1-0.aws.cloud.qdrant.io


In [18]:
# 임베딩 함수
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# TODO: 팀 프로젝트에 맞는 컬렉션 이름으로 변경하세요
# 예시: "team1_healthcare_docs", "team2_legal_docs" 등
collection_name = "SMU_2026_2_time"

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

if existing_collection:
    print(f"컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == 'y':
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("컬렉션이 삭제되었습니다.")

        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"컬렉션 '{collection_name}' 생성 완료")
    else:
        print("기존 컬렉션을 사용합니다.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")

# 벡터스토어 생성
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Child chunk 추가
uuids = [str(uuid4()) for _ in range(len(child_docs))]
vectorstore.add_documents(documents=child_docs, ids=uuids)

print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")

컬렉션 'SMU_2026_2_time' 생성 완료

914개의 Child chunk가 Qdrant Cloud에 추가되었습니다.


## 4. Parent Document 저장 (Docstore)

In [19]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

Docstore에 103개의 Parent 문서 저장 완료

Docstore 키 예시: ['page_1', 'page_2', 'page_3', 'page_4', 'page_5']


## 5. Parent Document Retriever 구현

In [20]:
from typing import List

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 2):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        # 1. Vectorstore에서 child chunk 검색
        child_results = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3) -> List[Document]:
        """비교용: Child chunk 직접 반환"""
        return self.vectorstore.similarity_search(query, k=k)

# Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=2
)

print("✓ Parent Document Retriever 생성 완료")

✓ Parent Document Retriever 생성 완료


## 6. 검색 테스트

**TODO: 팀 문서에 맞는 질문으로 변경하여 검색 테스트를 진행하세요**

In [21]:
# TODO: 팀 문서에 맞는 검색 질문을 작성하세요
query = "나는 AI모빌리티공학과 2학년이고, 22학점 수강신청을 할거야. 나한테 맞는 강의 추천해줘. 전공은 2학년전공 다 들을거야."

print(f"검색 쿼리: {query}\n")
print("="*80)

# Child chunk 검색
print("\n[1] Child Chunk 검색 결과")
print("-"*80)
child_results = parent_retriever.get_child_chunks(query, k=2)

for i, result in enumerate(child_results, start=1):
    print(f"\nChunk {i}:")
    print(f"  페이지: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용: {result.page_content}")

# Parent document 검색
print("\n" + "="*80)
print("\n[2] Parent Document 검색 결과")
print("-"*80)
parent_results = parent_retriever.invoke(query)

for i, result in enumerate(parent_results, start=1):
    print(f"\nPage {i}:")
    print(f"  페이지 번호: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용 미리보기: {result.page_content[:300]}...")

검색 쿼리: 나는 AI모빌리티공학과 2학년이고, 22학점 수강신청을 할거야. 나한테 맞는 강의 추천해줘. 전공은 2학년전공 다 들을거야.


[1] Child Chunk 검색 결과
--------------------------------------------------------------------------------

Chunk 1:
  페이지: 55
  Parent ID: page_55
  길이: 182자
  내용: 수강신청1일차주전공외  7   2 1전선 HBEA0026 인공지능기초                3   1   2                    월1,2,3(I409)          1   김혜윤                                                                  수강제한

Chunk 2:
  페이지: 85
  Parent ID: page_85
  길이: 182자
  내용: 수강신청1일차주전공외  1   2 1전선 HBEA0026 인공지능기초                3   1   2                    월1,2,3(I409)          1   김혜윤                                                                  수강제한


[2] Parent Document 검색 결과
--------------------------------------------------------------------------------

Page 1:
  페이지 번호: 55
  Parent ID: page_55
  길이: 5528자
  내용 미리보기:                2026학년도2학기학과별시간표(공지용)


공과대학시스템반도체공학과
                                  실습             강의시간    분 No 학년이수  학수번호      교과목명     학점이론     교양영역        

## 7. RAG 시스템 구현

**TODO: 시스템 프롬프트를 팀 문서에 맞게 수정하세요**

In [22]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display

llm = init_chat_model("gpt-5.4-mini")

# TODO: 시스템 프롬프트를 팀 문서 도메인에 맞게 수정하세요
# 예시: "당신은 의료 전문가입니다.", "당신은 법률 전문가입니다." 등
template = """
당신은 상명대에서 근무하는 교직원입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.
또한, 답변에 참고한 문서의 출처와 페이지 번호를 명시하세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

def rag_with_parent_retriever(question: str) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    """
    # 1. 문서 검색
    retrieved_docs = parent_retriever.invoke(question)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content

print("✓ RAG 시스템 준비 완료")

✓ RAG 시스템 준비 완료


## 8. RAG 시스템 테스트

**TODO: 팀 문서에 맞는 다양한 질문으로 RAG 시스템을 테스트하세요**

In [23]:
# TODO: 팀 문서에 맞는 질문들을 작성하세요
questions = [
    "나는 AI모빌리티공학과 2학년인데 내가 들을수 있는 강의 알려줘.",
    "학과 2학년 전공은 다 들을거고 빈 시간에 교양을 들을거야. 점심을 먹을수있게 연달아서 있는건 제외해줘",
    "공강을 2일정도 있게끔 만들어줘."
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))


질문: 나는 AI모빌리티공학과 2학년인데 내가 들을수 있는 강의 알려줘.



AI모빌리티공학과 **2학년** 학생이 수강할 수 있는 과목은 아래와 같습니다.  
(참고로, **2학년 전공필수/전공선택** 과목 기준으로 확인했습니다.)

### 2학년 수강 가능 과목
1. **회로이론및실습**  
   - 학수번호: **HBAM0012**
   - 구분: **1전선**
   - 학점: 3
   - 시간: **목 1, 2, 3**  
   - 강의실: **C415**

2. **모빌리티동역학**  
   - 학수번호: **HBAM0013**
   - 구분: **1전선**
   - 학점: 3
   - 시간: **목 8, 9, 10**  
   - 강의실: **I105**

3. **기계학습**  
   - 학수번호: **HBAM0014**
   - 구분: **1전선**
   - 학점: 3
   - 시간: **수 5, 6, 7**  
   - 강의실: **C415**

4. **모빌리티비전시스템**  
   - 학수번호: **HBAM0015**
   - 구분: **1전선**
   - 학점: 3
   - 시간: **수 1, 2, 3**  
   - 강의실: **N202**

### 참고
- 위 과목들은 **공과대학 AI모빌리티공학과 2학년** 시간표에 기재된 과목입니다.
- 다만, **세부 수강 가능 여부는 학적 상태, 선수과목 이수 여부, 분반 상황**에 따라 달라질 수 있습니다.

### 출처
- **3._2026-2학기_개설학과별_시간표_2026.8.19._기준.pdf, p.60**


질문: 학과 2학년 전공은 다 들을거고 빈 시간에 교양을 들을거야. 점심을 먹을수있게 연달아서 있는건 제외해줘



2학년 전공을 모두 수강하되, **점심시간에 연달아 수업이 이어지는 교양은 제외**해서 볼게요.  
다만, 주어진 자료에는 **각 학과 2학년 전공 시간표가 모두 포함되어 있지 않고**, 교양 과목 시간표만 일부 확인됩니다. 그래서 현재 자료로는 **전공 수업과 겹치지 않으면서 점심시간이 비는 교양 과목**을 정확히 골라드리기 어렵습니다.

현재 참고자료에서 확인 가능한 교양 과목 중, **연달아 붙어 있어 점심시간 확보가 어려운 과목**은 다음과 같습니다.

- **월 2,3,4**: 전공별프로젝트산출물실습caps (HBLF7834) — 73p
- **목 6,7,8**: AI시대의감정읽기와융합적자기성찰(캡스톤) (HBLF7835) — 73p
- **수 2,3,4**: 캡스톤과창의적문제해결 (HBLF7836) — 73p
- **목 6,7,8**: AI씽킹:사고와소통,그리고디자인 (HBLF7837) — 73p
- **월 6,7,8**: 세계시민성과DEI스토리랩 (HBLF7841) — 73p
- **월 3,4,5**: 상상력과문학 (HBLG1007) — 73p
- **목 2,3,4**: 법학의세계 (HBLG1009) — 73p
- **화 7,8**: 정보윤리와보안 (HBLG2016) — 73p
- **수 7,8,9**: 한국사회의이해(유학생) (HBLG2038) — 73p
- **금 2,3,4**: 한국문화입문 (HBLG2040) — 73p
- **금 5,6,7**: 충남의이해 (HBLG2050) — 73p

반면, **점심시간을 비교적 비워둘 가능성이 있는 교양**은 이런 형태입니다.

- **화 3,4** 농구 (HBLG2020) — 73p
- **수 5,6** 유머화법 (HBLG2033) — 73p
- **화 7,8** 정보윤리와보안 (HBLG2016) — 73p
- **금 4,5** 에너지와녹색산업 (HBLG1014) — 73p
- **토 0,1,2 / 토 3,4,5** 과목들 — 73p  
  - 나도콘텐츠크리에이터:AI로배우는창작 (HBLH7843)
  - 아이디어를현실로-바이브코딩 (HBLH7844)

하지만 최종적으로는 **사용자님의 2학년 전공 시간표**를 알아야 “전공을 다 듣고 남는 빈 시간”을 정확히 계산할 수 있습니다.  
**학과명이나 2학년 전공 시간표**를 보내주시면, 제가 바로

1. 전공 수업 시간 제외  
2. 점심시간 연속 수업 제외  
3. 남는 교양 후보만 추려서  

깔끔하게 정리해드릴게요.

**참고 문서 출처**
- 3._2026-2학기_개설학과별_시간표_2026.8.19._기준.pdf, **p.73**
- 3._2026-2학기_개설학과별_시간표_2026.8.19._기준.pdf, **p.10**


질문: 공강을 2일정도 있게끔 만들어줘.



가능합니다. 주어진 시간표 기준으로 **공강이 2일 정도 생기도록** 수강 조합을 추천드리면, 한 전공 안에서 수업을 고를 때 **요일이 겹치지 않는 과목 위주**로 선택하면 됩니다.

## 1) 디지털만화영상전공 기준 추천
아래처럼 고르면 **월/화/수/목 중 일부를 비우고**, 비교적 공강을 만들기 쉽습니다.

- **만화애니메이션페인팅**  
  - 화 1,2,3 / 금 1,2,3 / 금 5,6,7 중 택1
- **3D애니메이션Ⅰ**  
  - 수 8,9,10 또는 금 5,6,7 중 택1
- **디지털영상편집기초**  
  - 수 5,6,7 또는 금 1,2,3 중 택1
- **AI 콘텐츠기획&제작**  
  - 화 1,2,3

이 중에서 예를 들어  
- **3D애니메이션Ⅰ(수 8,9,10)**  
- **디지털영상편집기초(금 1,2,3)**  
- **AI 콘텐츠기획&제작(화 1,2,3)**  
조합을 보면, 수업이 **화/수/금**에 몰리므로 **월/목 2일 공강**을 만들 수 있습니다.

## 2) AI미디어콘텐츠전공 기준 추천
이 전공은 **월·화·수·목·금에 고르게 분포된 과목이 많아서**, 조합을 잘 선택하면 공강 2일을 만들 수 있습니다.

예를 들어:
- **AI와예술**: 월 2,3,4 또는 화 1,2,3
- **3D컴퓨터그래픽스Ⅰ(SW)**: 목 1,2,3 또는 목 5,6,7
- **GUI디자인(PBL)**: 목 1,2,3
- **영상콘텐츠(PBL)**: 화 1,2,3

이 중에서 예를 들어  
- **AI와예술(화 1,2,3)**  
- **3D컴퓨터그래픽스Ⅰ(SW)(목 1,2,3)**  
- **영상콘텐츠(PBL)(화 1,2,3)** 는 겹치므로 같이 듣기 어렵고,  
대신  
- **AI와예술(월 2,3,4)**  
- **3D컴퓨터그래픽스Ⅰ(SW)(목 1,2,3)**  
- **GUI디자인(PBL)(목 1,2,3)** 는 목이 겹치므로 조정이 필요합니다.

따라서 이 전공에서는 **월/수/금 중 2일을 비우고, 화/목에 집중 배치**하는 방식이 가장 현실적입니다.  
예를 들어  
- **AI와예술(월 2,3,4)**  
- **3D컴퓨터그래픽스Ⅰ(SW)(목 1,2,3)**  
- **전공체험(AI미디어콘텐츠전공)(화 7,8,9)**  
처럼 조합하면 **수/금 2일 공강**을 만들 수 있습니다.

## 3) 정리
공강 2일을 만들고 싶다면, 핵심은:
1. **같은 요일에 여러 과목을 몰아 넣고**
2. **아예 비우는 요일을 2개 정하는 것**입니다.

### 가장 추천하는 방향
- **디지털만화영상전공**: 월·목 공강 또는 화·수 공강 형태가 비교적 쉬움
- **AI미디어콘텐츠전공**: 수·금 공강 또는 월·금 공강 형태가 가능

원하시면 제가 바로  
**“월/목 공강 2일”**, **“수/금 공강 2일”**처럼  
원하는 공강 요일에 맞춰 **실제로 겹치지 않는 시간표 조합**으로 짜드릴게요.

**출처**
- 「3._2026-2학기_개설학과별_시간표_2026.8.19._기준.pdf」, p.30  
- 「3._2026-2학기_개설학과별_시간표_2026.8.19._기준.pdf」, p.34

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] PDF 문서 선정 및 로딩 완료
- [ ] Child Chunk 생성 완료
- [ ] Qdrant Cloud에 데이터 저장 완료
- [ ] Parent Document Retriever 구현 완료
- [ ] 검색 테스트 완료 (Child vs Parent 비교)
- [ ] RAG 시스템 구현 완료
- [ ] 최소 3개 이상의 질문으로 테스트 완료
- [ ] 시스템 프롬프트 도메인에 맞게 수정 완료

---

## 추가 개선 아이디어

1. **청킹 전략 최적화**: chunk_size와 chunk_overlap 조정
2. **검색 개수 조정**: retriever의 k 값 변경
3. **프롬프트 개선**: 더 구체적인 답변 형식 지정
4. **메타데이터 활용**: 날짜, 카테고리 등 추가 필터링
5. **하이브리드 검색**: 키워드 + 벡터 검색 결합